# Books Data retrieval

In [ ]:
import requests

def search_books(query):
    # Search for books by title or author
    url = f"https://gutendex.com/books?search={query}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        print("Error fetching data")
        return []

# Example: Search for books by "Kafka"
books = search_books("Kafka")

for book in books:
    print(f"Title: {book['title']}")
    print(f"ID: {book['id']}")
    # Print the link to the plain text file
    print(f"Text Link: {book['formats'].get('text/plain; charset=utf-8')}")
    print("-" * 20)

In [13]:
import requests
data=requests.get("https://gutendex.com/books/?page=2").json()
x=len(data["results"])
y=data["count"]





In [16]:
import math
data=[]
iters=math.ceil(y/x)
for i in range(1,iters+1):
    temp=requests.get(f"https://gutendex.com/books/?page={i}").json()
    data.extend(temp["results"])




ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [17]:
import json

with open("data.json","w") as f:
    json.dump(data,f,indent=4)

# User data generation

In [ ]:
import json
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

def load_books_from_json(filepath):
    """Loads JSON and prepares a DataFrame with flattened authors and subjects."""
    with open(filepath, 'r', encoding='utf-8') as f:
        books_data = json.load(f)
    
    processed_books = []
    
    for book in books_data:
        # Extract Author Name (First one)
        authors = book.get('authors', [])
        author_name = authors[0]['name'] if authors else "Unknown"
        
        # Flatten subjects list for easier checking later
        subjects_list = book.get('subjects', [])
        
        # Simulate Length (2 to 20 hours)
        estimated_length_min = random.randint(120, 1200)
        
        processed_books.append({
            "book_id": book.get('id'),
            "title": book.get('title'),
            "author": author_name,
            "subjects": subjects_list,  # Keep as list
            "download_count": book.get('download_count', 0),
            "total_length_minutes": estimated_length_min
        })
        
    return pd.DataFrame(processed_books)

def get_weighted_book_selection(df_books, fav_authors, fav_genre, n_select=1):
    """
    Selects books based on user preferences.
    Weights: 
    - 50% chance: Pick from Favorite Author
    - 30% chance: Pick from Favorite Genre
    - 20% chance: Pick Random (Popular)
    """
    
    # 1. Identify Candidate Books
    # We create masks (True/False lists)
    is_fav_author = df_books['author'].isin(fav_authors)
    
    # Check if fav_genre is in the book's subject list (a bit slower but necessary)
    is_fav_genre = df_books['subjects'].apply(lambda x: fav_genre in x if x else False)
    
    # 2. Assign Weights to every book in the library
    # Base weight = download count (Popularity bias)
    weights = df_books['download_count'].fillna(1) + 1.0
    
    # Boost weights based on preferences
    weights[is_fav_author] *= 50  # Massive boost for fav author
    weights[is_fav_genre] *= 20   # Big boost for fav genre
    
    # Normalize weights
    weights = weights / weights.sum()
    
    # 3. Sample
    return df_books.sample(n=n_select, weights=weights, replace=True)


def simulate_interactions_advanced(n_users=500):
    print("🚀 Loading Books Data...")
    books_file = "data.json"
    
    if not os.path.exists(books_file):
        print(f"❌ File {books_file} not found. Please save your JSON data first.")
        return

    df_books = load_books_from_json(books_file)
    print(f"📚 Loaded {len(df_books)} books.")

    # Get distinct lists for persona generation
    all_authors = df_books['author'].unique()
    all_subjects = list(set([s for sublist in df_books['subjects'] for s in sublist]))
    
    interactions = []
    print(f"🤖 Simulating {n_users} users (this might take a moment)...")

    for user_id in range(1, n_users + 1):
        # --- 1. CREATE PERSONA ---
        # User loves 2 specific authors
        my_fav_authors = np.random.choice(all_authors, size=2, replace=False)
        
        # User loves 1 specific genre
        my_fav_genre = random.choice(all_subjects) if all_subjects else "Fiction"
        
        # Determine Activity Level (Casual vs Power User)
        n_books = random.randint(5, 40)
        
        # --- 2. SELECT BOOKS ---
        # We use the weighted function to pick books relevant to this user
        my_books = get_weighted_book_selection(df_books, my_fav_authors, my_fav_genre, n_select=n_books)
        
        # --- 3. GENERATE INTERACTION DETAILS ---
        for _, book in my_books.iterrows():
            
            # Check matches
            match_author = book['author'] in my_fav_authors
            match_genre = my_fav_genre in book['subjects']
            
            # --- CALCULATE "PROBABILITY TO FINISH" ---
            if match_author and match_genre:
                finish_prob = 0.95
                base_rating = 5
            elif match_author:
                finish_prob = 0.85
                base_rating = 4
            elif match_genre:
                finish_prob = 0.70
                base_rating = 3.5
            else:
                # Random book they stumbled upon
                finish_prob = 0.30 
                base_rating = 2.5

            # --- SIMULATE OUTCOME ---
            is_completed = random.random() < finish_prob
            total_time = book['total_length_minutes']
            
            if is_completed:
                status = "Completed"
                # Read time: 90% to 110% of total length
                read_time = int(total_time * random.uniform(0.9, 1.1))
                # Rating: Skewed towards the base_rating but with variance
                rating = min(5, max(1, int(np.random.normal(base_rating, 0.5))))
            else:
                status = "Abandoned"
                # Read time: 5% to 40% of length
                read_time = int(total_time * random.uniform(0.05, 0.40))
                # Rating: Lower for abandoned books
                rating = min(5, max(1, int(np.random.normal(base_rating - 1.0, 0.8))))

            # --- APPEND ---
            interactions.append({
                "user_id": user_id,
                "book_id": book['book_id'],
                "rating": rating,
                "read_time_minutes": read_time,
                "total_book_minutes": total_time,
                "status": status,
                "is_fav_author_match": int(match_author), # Helpful for debugging/analysis
                "is_fav_genre_match": int(match_genre)
            })

    # Save
    df_out = pd.DataFrame(interactions)
    # Remove duplicates (user reading same book twice)
    df_out = df_out.drop_duplicates(subset=['user_id', 'book_id'])
    
    os.makedirs("data/raw", exist_ok=True)
    output_path = "interactions.csv"
    df_out.to_csv(output_path, index=False)
    
    print(f"✅ Generated {len(df_out)} interactions.")
    print(f"💾 Saved to {output_path}")
    print(df_out.head())

if __name__ == "__main__":
    simulate_interactions_advanced()